# Bài 10 — Nâng cao

3 chủ đề mở rộng: phát hiện vật thể nhỏ (SAHI), làm việc với dataset, và đánh giá model bằng metrics.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [1]:
!pip install -q supervision ultralytics "supervision[assets]"

In [2]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 17:15:56] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


  0%|          | 0/35345757 [00:00<?, ?it/s]

vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## 10.1. InferenceSlicer — phát hiện vật thể nhỏ (SAHI)

Ảnh flycam/độ phân giải lớn, vật thể bé xíu → cắt ảnh thành ô, detect từng ô, gộp lại. So sánh trực quan **detect thường vs qua slicer** trên cửa sổ (ghép ngang như Bài 3.2).

In [5]:
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

from display import show_frame, close_windows

model = YOLO("yolov8n.pt")
image = next(sv.get_video_frames_generator("vehicles.mp4"))


def callback(image_slice: np.ndarray) -> sv.Detections:
    result = model(image_slice, verbose=False)[0]
    return sv.Detections.from_ultralytics(result)


slicer = sv.InferenceSlicer(
    callback=callback,
    slice_wh=(640, 640),
    overlap_wh=(128, 128),
)

# So sánh: detect bình thường (1 lần) vs qua slicer (nhiều ô nhỏ)
detections_normal = sv.Detections.from_ultralytics(model(image, verbose=False)[0])
detections_sliced = slicer(image)

box_annotator = sv.BoxAnnotator(thickness=2)
left = box_annotator.annotate(image.copy(), detections_normal)
right = box_annotator.annotate(image.copy(), detections_sliced)
cv2.putText(left, f"Binh thuong: {len(detections_normal)}", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
cv2.putText(right, f"Qua Slicer: {len(detections_sliced)}", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

compare = np.hstack([left, right])
show_frame(compare, window_name="Binh thuong vs InferenceSlicer", wait=0)
close_windows()

>  Với ảnh xe cỡ thường (như `vehicles.mp4`), 2 số này thường xấp xỉ nhau vì xe đã đủ lớn. `InferenceSlicer` chỉ thực sự tỏa sáng khi vật thể **rất nhỏ so với ảnh** (ảnh flycam chụp từ độ cao lớn, đếm người/xe li ti). Lưu ý: slicer chạy model nhiều lần (1 lần mỗi ô) nên **chậm hơn hẳn** — chỉ dùng khi cần thiết.

## 10.2. DetectionDataset — làm việc với dataset

 Cell này cần bạn có sẵn 1 dataset định dạng YOLO (thư mục `images/`, `labels/`, file `data.yaml`) — ví dụ tải từ [Roboflow Universe](https://universe.roboflow.com). Sửa lại đường dẫn cho khớp máy bạn.

In [6]:
import supervision as sv
from display import show_frame, close_windows

#  Đổi đường dẫn cho khớp dataset thật của bạn
ds = sv.DetectionDataset.from_yolo(
    images_directory_path="dataset/images",
    annotations_directory_path="dataset/labels",
    data_yaml_path="dataset/data.yaml",
)
print("So anh trong dataset:", len(ds))

# Chuyển đổi format: ds.as_coco(...), ds.as_pascal_voc(...)
# Chia tập: train_ds, test_ds = ds.split(split_ratio=0.8)

# Duyệt xem dataset bằng cửa sổ: bấm phím lật ảnh, Q thoát
box_annotator = sv.BoxAnnotator()
for _, image, gt in ds:
    if not show_frame(box_annotator.annotate(image.copy(), gt), wait=0):
        break
close_windows()

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/data.yaml'

## 10.3. Metrics — đánh giá model

Cần có `test_ds` (tập test với ground-truth annotation) từ mục 10.2.

In [ ]:
from supervision.metrics import MeanAveragePrecision, F1Score

map_metric = MeanAveragePrecision()
for _, image, gt in test_ds:
    pred = sv.Detections.from_ultralytics(model(image, verbose=False)[0])
    map_metric.update(pred, gt)

print(map_metric.compute())   # mAP50, mAP50-95...

## Checkpoint Bài 10

- 10.1: cửa sổ so sánh 2 kết quả detect thường vs qua slicer cạnh nhau, số lượng object hiện rõ trên mỗi bên
- 10.2 (nếu có dataset): duyệt được ảnh + ground-truth box bằng phím, Q thoát
- 10.3 (nếu có dataset): in ra được mAP50/mAP50-95 của model trên tập test
